<a href="https://colab.research.google.com/github/hiiamlars/analytical_flexibility_llm_reviews/blob/main/data_generation/llm_review_judgments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup

In [1]:
# @title Dependencies

# Installations
!pip install -q -U transformers peft accelerate bitsandbytes huggingface_hub
!pip install -q pandas tqdm sentencepiece

# First-party imports
from google.colab import drive
import google.colab.userdata
import random
import json
import os
import asyncio
import math
import concurrent.futures
import functools

# Third-party imports
import pandas as pd
from tqdm.auto import tqdm
from huggingface_hub import login
import tqdm.asyncio
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
from peft import PeftModel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.5/645.5 kB 21.6 MB/s eta 0:00:00


In [2]:
# @title Path definitions

# Mount GDrive
drive.mount('/content/drive')

# Set working directory
WORKING_DIR = "/content/drive/MyDrive/Analytical_flexibility_llm_reviews"

# Create working directory
os.makedirs(WORKING_DIR, exist_ok=True)
print(f"Ensured working directory exists at: {WORKING_DIR}.")

# Define and create DATASET_DIR to load the paper content
DATASET_DIR = os.path.join(WORKING_DIR, "llm_reviews_processed")
os.makedirs(DATASET_DIR, exist_ok=True)
print(f"Ensured dataset directory exists at: {DATASET_DIR}.")

# Define and create the RAW_DIR for the llm reviews (and their logs)
RAW_DIR = os.path.join(WORKING_DIR, "llm_reviews_judged")
os.makedirs(RAW_DIR, exist_ok=True)
print(f"Ensured raw directory exists at: {RAW_DIR}.")

# Define the full result path (final Parquet output)
RESULTS_PATH = os.path.join(RAW_DIR, "llm_reviews_evaluated.parquet")
# Define the full result path for intermediate JSONL storage
RESULTS_JSON_PATH = os.path.join(RAW_DIR, "llm_reviews_evaluated.jsonl")

# Define the full log path (final Parquet output)
LOG_PATH = os.path.join(RAW_DIR, "llm_reviews_log.parquet")
# Define the full log path for intermediate JSONL storage
LOG_JSON_PATH = os.path.join(RAW_DIR, "llm_reviews_log.jsonl")

Mounted at /content/drive
Ensured working directory exists at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews.
Ensured dataset directory exists at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews_processed.
Ensured raw directory exists at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews_judged.


In [3]:
# @title Setup definitions

### --- Settings --- ###

SIMULATION_ACTIVE = False # SIMULATION_ACTIVE = False activates the model
MAX_CONCURRENT_CALLS = 8

# Subsetting dataset content
SUBSETTING_ACTIVE = True # SUBSETTING_ACTIVE = True activates the data subsetting
NUM_SUBSET = 50

### --- API/Client Settings --- ###
MAX_RETRIES = 3
RETRY_DELAY = 3

# Error messages
JUDGE_GENERATION_FAILURE = "Error: Failed to generate judgement."
UNKNOWN_ERROR_STATE = "Error: Unknown client error."
PARSING_ERROR = "Error: Failed to parse judgement from JSON to single values"

# Error probabilities for simulation mode
ERROR_PROB_LLM = 0.15 # random
ERROR_PROB_PARSING = 0.15 # random

### --- Content --- ####

TASK_DESCRIPTION_HEADER = '''###Task Description: You are an expert in evaluating peer review comments with respect to different aspects. These aspects are aimed to maximize the utilization of the review comments for the authors. The primary purpose of the review is to help/guide authors in improving their drafts. Keep this in mind while evaluating the review point. Whenever you encounter a borderline case, think: “Will this review point help authors improve their draft?”. There is no correlation between the aspect score and the length of the review point.'''

ASPECTS_NO_EXAMPLES = {
"actionability":
'''
**Actionability**

**Definition:** Measures the level of actionability in a review point. We evaluate actionability based on two criteria:

1. **Explicit vs. Implicit**:
   - **Explicit:** Actions or suggestions that are direct or apparent. Authors can directly identify modifications they should apply to their draft. Clarification questions should be treated as explicit statements if they give a direct action.
   - **Implicit:** Actions that need to be inferred from the comment. This includes missing parts that need to be added. Authors can deduce what needs to be done after reading the comment.

2. **Concrete vs. Vague**:
   - **Concrete:** Once the action is identified, the authors know exactly what needs to be done and how to apply the action.
   - **Vague:** After identifying the action, the authors still don’t know how to carry out this action.

**Importance:** It’s more important for actions to be concrete so that authors know how to apply them. It's also preferred for actions to be stated directly rather than inferred.

**Actionability Scale (1-5):**

1. **1: Unactionable**
   - **Definition:** The comment lacks meaningful information to help authors improve the paper. Authors do not know what they should do after reading the comment.

2. **2: Borderline Actionable**
   - **Definition:** The comment includes an implicitly stated action or an action that can be inferred. However, the action itself is vague and lacks detail on how to apply it.

3. **3: Somewhat Actionable**
   - **Definition:** The comment explicitly states an action but is vague on how to execute it.

4. **4: Mostly Actionable**
   - **Definition:** The comment implicitly states an action but concretely states how to implement the inferred action.

5. **5: Highly Actionable**
   - **Definition:** The comment contains an explicit action and concrete details on how to implement it. Authors know exactly how to apply it.
''',

"grounding_specificity": '''

**Grounding Specificity**

**Definition:** Measures how explicitly a review comment refers to a specific part of the paper and how clearly it identifies the issue with that part. This helps authors understand what needs revision and why. Grounding specificity has two key components:

1. **Grounding:** How well the authors can identify the specific part of the paper being addressed.
   - **Weak Grounding:** The author can make an educated guess but cannot precisely identify the referenced part.
   - **Full Grounding:** The author can accurately pinpoint the section, table, figure, or unique aspect being addressed. This can be achieved through:
     - Literal mentions of sections, tables, figures, etc.
     - Mentions of unique elements of the paper.
     - General comments that clearly imply the relevant parts without explicitly naming them.

2. **Specificity:** How clearly the comment details what is wrong or missing in the referenced part. If external work is mentioned, it also evaluates whether specific examples are provided.

**Importance:** It's more important for the comment to be grounded than to be specific.

**Grounding Specificity Scale (1-5):**

1. **Not Grounded**
   - **Definition**: This comment is not grounded at all. It does not identify a specific area in the paper. The comment is highly unspecific.

2. **Weakly Grounded and Not Specific**
   - **Definition**: The authors cannot confidently determine which part the comment addresses. Further, the comment does not specify what needs to be addressed in this part.

3. **Weakly Grounded and Specific**
   - **Definition**: The authors cannot confidently determine which part the comment addresses. However, the comment clearly specifies what needs to be addressed in this part.

4. **Fully Grounded and Under-Specific**
   - **Definition**: The comment explicitly mentions which part of the paper it addresses, or it should be obvious to the authors. However, this comment does not specify what needs to be addressed in this part.

5. **Fully Grounded and Specific**
   - **Definition**: The comment explicitly mentions which part of the paper it addresses, and it is obvious to the authors. The comment specifies what needs to be addressed in this part.
''',

"verifiability":
'''
**Verifiability**

**Definition:** Evaluates whether a review comment contains a claim and, if so, how well that claim is supported using logical reasoning, common knowledge, or external references.

### **Step 1: Claim Extraction**

**Objective:**
Determine whether the given text contains a claim (i.e., an opinion, judgment, or suggestion) or consists solely of factual statements that require no verification.

**Claim Definition:**
A statement is considered a claim if it falls into one or more of the following categories:
- **Subjective opinions or disagreements** (e.g., criticism of an experimental choice).
- **Suggestions or requests for changes** (e.g., recommending removal, addition, or discussion).
- **Judgments about the paper** (e.g., stating something is unclear, not well-written, or lacks detail).
- **Deductions or inferred observations** that go beyond merely stating facts.
- **Statements requiring justification** to be understood or accepted.


**Normal Statements ("No Claim")**
A statement is classified as "X" if it:
- Describes facts without suggesting changes.
- Makes general statements about the paper without an opinion.
- Presents objective, verifiable facts that require no justification.
- Asks for clarifications or general questions.
- States logical statements or directly inferable information.
- Makes positive claims (e.g., *“The paper is well-written”*), as these do not help improve the work.

---

### **Step 2: Verifiability Verification**

**Objective:**
Assess how well a claim is verified by examining the reasoning, common knowledge, or external references provided. The purpose is to ensure that the review comment helps the authors improve their work.

**Verification Methods:**
A claim is considered verifiable if supported by one or more of the following:
- **Logical reasoning** – A clear explanation of why the claim is valid.
- **Common knowledge** – Reference to well-accepted practices or standards.
- **External references** – Citation of relevant literature, data, or sources.

**Verifiability Scale (1–5 & X):**

1. **1: Unverifiable**
   - The comment contains a claim without any supporting evidence or justification.
2. **2: Borderline Verifiable**
   - Some support is provided, but it is vague, insufficient, or difficult to follow.
3. **3: Somewhat Verifiable**
   - The claim has some justification but lacks key elements (e.g., examples, references).
4. **4: Mostly Verifiable**
   - The claim is well-supported but has minor gaps in explanation or references.
5. **5: Fully Verifiable**
   - The claim is thoroughly supported by explicit, sufficient, and robust evidence, such as:
     - Clear reasoning and precise explanations.
     - Specific references to external works.
     - Logical and unassailable common-sense arguments.
6. **X: No Claim**
- The comment contains only factual, descriptive statements without claims, opinions, or suggestions.

---

### **Instructions for Evaluation:**
1. **Extract Claims:** Determine whether the review comment contains a claim or is a normal statement. If there is no claim, score it as "X"
2. **Assess Verifiability:** If a claim exists, score it based on how well it is justified from 1 to 5.
'''
,

"helpfulness" : '''
**Helpfulness**

**Definition:** Assign a subjective score to reflect the value of the review comment to the authors. Helpfulness is rated on a scale from 1 to 5, with the following definitions:

1. **1: Not Helpful at All**
   - **Definition:** The comment fails to identify meaningful weaknesses or suggest improvements, leaving the authors with no actionable feedback.

2. **2: Barely Helpful**
   - **Definition:** The comment identifies a weakness or improvement area but is vague, lacks clarity, or provides minimal guidance, making it only slightly beneficial for the authors.

3. **3: Somewhat Helpful**
   - **Definition:** The comment identifies weaknesses or areas for improvement but is incomplete or lacks depth. While the authors gain some insights, the feedback does not fully address their needs for improving the draft.

4. **4: Mostly Helpful**
   - **Definition:** The comment provides clear and actionable feedback on weaknesses and areas for improvement, though it could be expanded or refined to be fully comprehensive and impactful.

5. **5: Highly Helpful**
   - **Definition:** The comment thoroughly identifies weaknesses and offers detailed, actionable, and constructive suggestions that empower the authors to significantly improve their draft.
'''
}

INSTRUCTION_SCORE_ONLY_PROMPT_TAIL = '''
###Instruction:
Evaluate the review based on the given definitions of the aspect(s) above. Output only the score. The possbile values for scores are 1-5 and X.

Generate the output in JSON format with the following format:

   "actionability_label": "",
   "grounding_specificity_label": "",
   "verifiability_label": "",
   "helpfulness_label": ""


###Review Point:
{review_point}''' ### According to the repo code, but with deviation from the examplary prompt in the paper

### --- API/Client --- ###

# HF token key
hf_token = google.colab.userdata.get('HF_TOKEN')
login(token=hf_token)


In [4]:
# Setup names and config
BASE_MODEL = "meta-llama/Llama-3.1-8B-Instruct"
ADAPTER_MODEL = "boda/RevUtil_Llama-3.1-8B-Instruct_score_only"

# Conditionally load model and tokenizer based on SIMULATION_ACTIVE
if not SIMULATION_ACTIVE:

    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=hf_token)

    # Set pad token
    tokenizer.pad_token = tokenizer.eos_token

    # Configure 4-bit quantization
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    # Load Base model
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=quantization_config,
        device_map="auto",
        attn_implementation="sdpa",
        token=hf_token,
        trust_remote_code=True,
    )

    # Attach the RevUtil Adapter
    model = PeftModel.from_pretrained(
        base_model,
        ADAPTER_MODEL,
        token=hf_token
    )

    # Set model to eval mode
    model.eval()

    # Configure model with pad token
    model.config.pad_token_id = tokenizer.pad_token_id
    tokenizer.padding_side = "right"

else:
    print("SIMULATION_ACTIVE is True. Skipping model and tokenizer loading to save resources.")
    tokenizer = None
    model = None

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/734 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/168M [00:00<?, ?B/s]

In [5]:
# @title Data import

# Read processed llm reviews
df_reviews = pd.read_parquet(os.path.join(DATASET_DIR, "llm_reviews_segmented.parquet"))

# Check df_reviews
display(df_reviews.head(3))
display(df_reviews.shape)

,paper_id,model,temperature,reasoning_effort,chain_of_thought,note_taking,run_signature,llm_notes_1,llm_review_1,llm_notes_2,llm_review_2,extracted_weakness,segmented_comment,segmented_comment_id
0,author_2,gpt-5-mini-2025-08-07,NaN,low,,Faithful,bd301abd5453,Title: DiP-GNN: Discriminative Pre-Training of...,# Summary Of The Paper\nThis paper identifies ...,"Paper: ""DiP-GNN: Discriminative Pre-Training o...",# Summary Of The Paper\nThe paper proposes DiP...,Weaknesses\nStrengths\n- Clear problem identif...,"- Clear problem identification: the ""graph mis...",39f5a3306e4de696d42e604c9d34aa0b7b41f352903e37...
1,author_2,gpt-5-mini-2025-08-07,NaN,low,,Faithful,bd301abd5453,Title: DiP-GNN: Discriminative Pre-Training of...,# Summary Of The Paper\nThis paper identifies ...,"Paper: ""DiP-GNN: Discriminative Pre-Training o...",# Summary Of The Paper\nThe paper proposes DiP...,Weaknesses\nStrengths\n- Clear problem identif...,"- Simple, scalable method: joint generator/dis...",e0601ee55396e655c69cfaf2b7fae52d23c48fa943d1d8...
2,author_2,gpt-5-mini-2025-08-07,NaN,low,,Faithful,bd301abd5453,Title: DiP-GNN: Discriminative Pre-Training of...,# Summary Of The Paper\nThis paper identifies ...,"Paper: ""DiP-GNN: Discriminative Pre-Training o...",# Summary Of The Paper\nThe paper proposes DiP...,Weaknesses\nStrengths\n- Clear problem identif...,- Strong empirical gains: consistent improveme...,74e884b25c8ebb96215d4bedb943d10265f17a96242d7e...


(218, 14)

In [6]:
# @title Subsetting (if active)

if SUBSETTING_ACTIVE:
    df_reviews = df_reviews.head(NUM_SUBSET)
    print(f"Data subsetted to {NUM_SUBSET} rows.")
    display(df_reviews.head(3))
    display(df_reviews.shape)
else:
    print("Paper content subsetting is inactive.")

Data subsetted to 50 rows.


,paper_id,model,temperature,reasoning_effort,chain_of_thought,note_taking,run_signature,llm_notes_1,llm_review_1,llm_notes_2,llm_review_2,extracted_weakness,segmented_comment,segmented_comment_id
0,author_2,gpt-5-mini-2025-08-07,NaN,low,,Faithful,bd301abd5453,Title: DiP-GNN: Discriminative Pre-Training of...,# Summary Of The Paper\nThis paper identifies ...,"Paper: ""DiP-GNN: Discriminative Pre-Training o...",# Summary Of The Paper\nThe paper proposes DiP...,Weaknesses\nStrengths\n- Clear problem identif...,"- Clear problem identification: the ""graph mis...",39f5a3306e4de696d42e604c9d34aa0b7b41f352903e37...
1,author_2,gpt-5-mini-2025-08-07,NaN,low,,Faithful,bd301abd5453,Title: DiP-GNN: Discriminative Pre-Training of...,# Summary Of The Paper\nThis paper identifies ...,"Paper: ""DiP-GNN: Discriminative Pre-Training o...",# Summary Of The Paper\nThe paper proposes DiP...,Weaknesses\nStrengths\n- Clear problem identif...,"- Simple, scalable method: joint generator/dis...",e0601ee55396e655c69cfaf2b7fae52d23c48fa943d1d8...
2,author_2,gpt-5-mini-2025-08-07,NaN,low,,Faithful,bd301abd5453,Title: DiP-GNN: Discriminative Pre-Training of...,# Summary Of The Paper\nThis paper identifies ...,"Paper: ""DiP-GNN: Discriminative Pre-Training o...",# Summary Of The Paper\nThe paper proposes DiP...,Weaknesses\nStrengths\n- Clear problem identif...,- Strong empirical gains: consistent improveme...,74e884b25c8ebb96215d4bedb943d10265f17a96242d7e...


(50, 14)

## Output check

Original regex-based output checks from the authors `inference_utils.py` to enforce a clean final JSON-output structure: https://github.com/bodasadallah/RevUtil/blob/main/inference/inference_utils.py

In [4]:
import json
import re


known_keys = ['actionability_rationale', 'actionability_label', 'grounding_specificity_rationale', 'grounding_specificity_label', 'verifiability_rationale', 'verifiability_label', 'helpfulness_rationale', 'helpfulness_label']


def replace_category_names(text):
    category_mapping = {
        "1: Unverifiable": 1,
        "2: Borderline Verifiable": 2,
        "3: Somewhat Verifiable": 3,
        "4: Mostly Verifiable": 4,
        "5: Fully Verifiable": 5,
        "X: No Claim": "X",

        "1: Unactionable": 1,
        "2: Borderline Actionable": 2,
        "3: Somewhat Actionable": 3,
        "4: Mostly Actionable": 4,
        "5: Highly Actionable": 5,

        "1: Not Grounded": 1,
        "2: Weakly Grounded and Not Specific": 2,
        "3: Weakly Grounded and Specific": 3,
        "4: Fully Grounded and Under-Specific": 4,
        "5: Fully Grounded and Specific": 5,

        "1: Not Helpful at All": 1,
        "2: Barely Helpful": 2,
        "3: Somewhat Helpful": 3,
        "4: Mostly Helpful": 4,
        "5: Highly Helpful": 5
    }

    # Normalize dictionary for case-insensitive matching
    category_mapping_lower = {k.lower(): v for k, v in category_mapping.items()}
    partial_mapping_lower = {k.split(": ", 1)[-1].lower(): v for k, v in category_mapping.items()}

    def replace_match(match):
        matched_text = match.group(0).lower()  # Normalize case
        return str(category_mapping_lower.get(matched_text, partial_mapping_lower.get(matched_text, match.group(0))))

    # Replace full matches first (case-insensitive)
    pattern = re.compile(r'\b(?:' + '|'.join(map(re.escape, category_mapping_lower.keys())) + r')\b', re.IGNORECASE)
    text = pattern.sub(replace_match, text)

    # Replace partial matches (case-insensitive)
    pattern_partial = re.compile(r'\b(?:' + '|'.join(map(re.escape, partial_mapping_lower.keys())) + r')\b', re.IGNORECASE)
    text = pattern_partial.sub(replace_match, text)

    return text


expected_keys = [
    "actionability_rationale",
    "actionability_label",
    "grounding_specificity_rationale",
    "grounding_specificity_label",
    "verifiability_rationale",
    "verifiability_label",
    "helpfulness_rationale",
    "helpfulness_label"
]

import json
import re

import json

def extract_valid_json(text):
    label_keys = [
        "actionability_label",
        "grounding_specificity_label",
        "verifiability_label",
        "helpfulness_label"
    ]

    # Initialize result with None values
    result = {key: None for key in label_keys}

    # Match patterns like: "key": "1", "key": 1, or "key": "X"
    pattern = r'"(' + '|'.join(label_keys) + r')"\s*:\s*"?(X|\d+)"?'

    matches = re.findall(pattern, text)

    for key, val in matches:
        result[key] = str(val)  # Always store as string for valid JSON output

    return json.dumps(result, indent=2)



def escape_inner_quotes(text):
    """Finds specified rationale fields and escapes only internal double quotes."""
    fields = [
        "actionability_rationale",
        "grounding_specificity_rationale",
        "verifiability_rationale",
        "helpfulness_rationale"
    ]

    for field in fields:
        pattern = fr'("{field}"\s*:\s*")(.*?)("[\}},])'  # Escape closing brace
        matches = list(re.finditer(pattern, text, re.DOTALL))  # Find all matches first

        for match in reversed(matches):  # Process from last to first to avoid index shifting
            before, rationale, after = match.groups()
            escaped_rationale = rationale.replace('"', '\\"')  # Escape only inner quotes
            text = text[:match.start(2)] + escaped_rationale + text[match.end(2):]

    return text
def extract_dict(text):

    text = replace_category_names(text)  # Replace category names with numbers
    ## remove double spaces
    text = re.sub(' +', ' ', text)
    ## remove ``` from the text
    # text = text.replace('```', '')
    text = text.replace("-", "")  # Remove leading hyphens
    text = text.replace("\n", " ")  # Remove newlines
    text = text.replace("\\'", "'")  # Fix incorrectly escaped single quotes
    text = text.replace('\\"s', "'s")  # Fix incorrect escaped possessive 's
    text = text.replace("\\\\'", "\\\"")
    text = text.replace("\\\\", "\\")

    text = text.replace("[", "")  # Remove square brackets
    text = text.replace("]", "")  # Remove square brackets

    ############## For Prometheus2 #################
    # text = text.replace("[", '"')  # Replace single quotes with double quotes
    # text = text.replace("]", '"')  # Replace single quotes with double quotes

    ## if text begin with comma or space, remove it
    if text[0] == ',' or text[0] == ' ':
        text = text[1:]

    text = text.replace("\\\\", "\\") # Fix double backslashes
    dict_str  = ""
    if "```" in text:
        text = text + '#'
        match = re.search(r"```(?:json)?\s*(.*?)(```)?#", text, re.DOTALL)
        if match:
            text = match.group(0)
            ## remove the ```json  and ``` from the text
            text = text.replace('```json', '')
            text = text.replace('```', '')
            text = text.replace('#', '')

    text = text.strip()  # Remove leading and trailing whitespace

    if not text:
        return None

    ############# Comment for Prometheus2 ##############
    if text[0] != '{':
        text = '{' + text + '}'

    ################## cut the text if there is two newlines. This is for Prmetheus2 #########
    # if '\n\n' in text:
    #     halfs = text = text.split('\n\n', 1)
    #     text = halfs[0] if "actionability_label" in halfs[0] else halfs[1]

    if '{' not in text:
        text = '{' + text + '}'

    text = text.replace(" }  { ", ',')  # Remove newlines between dictionaries

    text = text.replace("\n", ' ')  # Remove newlines

    ############################# Prometheus2 ##########################
    # text = extract_valid_json(text)
    # text = text.replace('\\', '')  # Remove single quotes
    # print(f"Text after processing: {text} \n\n\n")

    ############ Some cases doesn't work with replacing the quotes, so trying both ways
    text2 = text
    try:
        text = text.replace("'", '"')  # Replace single quotes with double quotes
        text = escape_inner_quotes(text)  # Fix quotes inside rationale fields
        match = re.search(r'\{.*?\}', text, re.DOTALL)  # Extract first {...} block
        if match:
            dict_str = match.group()  # Get extracted dictionary string
        return json.loads(dict_str)  # Convert to Python dictionary safely
    except json.JSONDecodeError:
        print("Replacing quotes didn't work, trying without replacing quotes.")
        text = text2  # Revert to original text
        match = re.search(r'\{.*?\}', text, re.DOTALL)  # Extract first {...} block
        if match:
            dict_str = match.group()  # Get extracted dictionary string
        try:
            return json.loads(dict_str)  # Convert to Python dictionary safely
        except json.JSONDecodeError as e:
            print(f"Parsing error: {e}\nProblematic string: {dict_str}")
            return None

def extract_predictions(model_outputs):
    """
    Parses a list of model-generated texts to extract labels and returns a dictionary.

    :param model_outputs: List of strings containing model-generated text with labels.
    :return: List of dictionaries with extracted labels.
    """
    extracted_data = []

    for text in model_outputs:

        if 'outputs' in text.keys():
            text = text.outputs[0].text
        elif 'generated_text' in text.keys():
            text = text['generated_text']

        extracted_dict = extract_dict(text)
        if  not extracted_dict:
            extracted_data.append({
                'actionability_label': None,
                'grounding_specificity_label': None,
                'verifiability_label': None,
                'helpfulness_label': None,
                'actionability_rationale': None,
                'grounding_specificity_rationale': None,
                'verifiability_rationale': None,
                'helpfulness_rationale': None
            })
            continue

        parsed_result = {
            'actionability_label': str(extracted_dict.get('actionability_label', None)),
            'grounding_specificity_label':  str(extracted_dict.get('grounding_specificity_label', None)),
            'verifiability_label':  str(extracted_dict.get('verifiability_label', None)),
            'helpfulness_label':  str(extracted_dict.get('helpfulness_label', None)),
            ### rationale keys
            'actionability_rationale':  str(extracted_dict.get('actionability_rationale', None)),
            'grounding_specificity_rationale':  str(extracted_dict.get('grounding_specificity_rationale', None)),
            'verifiability_rationale':  str(extracted_dict.get('verifiability_rationale', None)),
            'helpfulness_rationale':  str(extracted_dict.get('helpfulness_rationale', None))
        }

        extracted_data.append(parsed_result)

    return extracted_data


## Functions

In [5]:
def build_prompt(review_point: str) -> str:
    """Constructs the full prompt as a single string for text_generation."""
    system_content_parts = []
    system_content_parts.append(TASK_DESCRIPTION_HEADER)

    for aspect_key, aspect_definition_text in ASPECTS_NO_EXAMPLES.items():
        if aspect_key == "grounding_specificity":
            display_aspect_name = "Grounding & Specificity"
        else:
            display_aspect_name = aspect_key.replace('_', ' ').title()
        system_content_parts.append(f"Aspect: {display_aspect_name} {aspect_definition_text}")

    instruction_part = INSTRUCTION_SCORE_ONLY_PROMPT_TAIL.split("###Review Point:")[0].strip()
    system_content_parts.append(instruction_part)

    system_message_content = "\n".join(system_content_parts)

    user_message_content = f"###Review Point:\n{review_point}"

    # Combine system and user content into a single string
    full_prompt_string = system_message_content + "\n" + user_message_content

    return full_prompt_string

In [6]:
import asyncio
import functools
import torch
import json

# @title ScoringClient

class ScoringClient:
    def __init__(self, local_model, local_tokenizer, simulate: bool = False, seed: int = 42):
        """
        Initialize ScoringClient for local model inference.
        This client handles both simulation and actual local inference, including retry logic.
        """
        self.simulate = simulate
        self.rng = random.Random(seed)
        self.local_model = local_model
        self.local_tokenizer = local_tokenizer
        self._executor = concurrent.futures.ThreadPoolExecutor(max_workers=MAX_CONCURRENT_CALLS) # Initialize ThreadPoolExecutor

        if not self.simulate:
            print(f"ScoringClient initialized for local model.")
        else:
            print("ScoringClient running in simulation mode.")

    async def score_batch_review_points(self, batch_data: list[dict]) -> list[dict]:
        """
        Scores a batch of review points by building prompts and calling the local model
        or returning simulated responses. Handles retry logic for the batch.
        Returns a list of dictionaries, where each dict is a result_entry.
        """
        prompts = [build_prompt(row['segmented_comment']) for row in batch_data]
        segmented_comment_ids = [row['segmented_comment_id'] for row in batch_data]

        # Prepare a list to store raw outputs from the LLM or simulation
        raw_llm_outputs_batch = []

        if self.simulate:
            await asyncio.sleep(0.1) # Simulate some delay
            for i, _ in enumerate(batch_data):
                random_val = self.rng.random()
                if random_val < ERROR_PROB_LLM:
                    raw_llm_outputs_batch.append(json.dumps({"error": JUDGE_GENERATION_FAILURE, "raw_llm_output": ""}))
                elif random_val < ERROR_PROB_LLM + ERROR_PROB_PARSING:
                    malformed_output = f"This is not valid JSON. {JUDGE_GENERATION_FAILURE} for {segmented_comment_ids[i]}"
                    raw_llm_outputs_batch.append(malformed_output)
                else:
                    simulated_output_dict = {
                        "actionability_label": str(self.rng.randint(1, 5)),
                        "grounding_specificity_label": str(self.rng.randint(1, 5)),
                        "verifiability_label": self.rng.choice([str(self.rng.randint(1, 5)), "X"]),
                        "helpfulness_label": str(self.rng.randint(1, 5))
                    }
                    raw_llm_outputs_batch.append(json.dumps(simulated_output_dict))
        else:
            for attempt in range(1, MAX_RETRIES + 1):
                try:
                    loop = asyncio.get_running_loop()
                    # Run the blocking local inference in the thread pool
                    raw_llm_outputs_batch = await loop.run_in_executor(
                        self._executor,
                        functools.partial(self._perform_batched_local_inference, prompts=prompts)
                    )
                    break # Break if successful
                except Exception as e:
                    print(f"[LLM ERROR - Local Inference] attempt={attempt} -> {e}", flush=True)
                    if attempt == MAX_RETRIES:
                        # For batch, if one fails, apply the error to all, or return partial results.
                        # For simplicity, returning error for all if the batched call fails.
                        raw_llm_outputs_batch = [json.dumps({"error": JUDGE_GENERATION_FAILURE, "raw_llm_output": str(e)})] * len(prompts)
                        break
                await asyncio.sleep(RETRY_DELAY)

        if not raw_llm_outputs_batch: # Fallback if for some reason no outputs were generated
             raw_llm_outputs_batch = [json.dumps({"error": UNKNOWN_ERROR_STATE, "raw_llm_output": "All retries failed or no outputs generated."})] * len(prompts)

        processed_results = []
        for i, raw_output_str in enumerate(raw_llm_outputs_batch):
            # Log parsing step
            print(f"[Parsing - ID: {segmented_comment_ids[i][:8]}...; Comment: {batch_data[i]['segmented_comment'][:50]}...]")

            # Convert raw_output_str (which might be JSON string from simulation or raw LLM output) to a dict
            try:
                parsed_output_dict = self._process_output_for_json(raw_output_str)
            except Exception as e:
                # If _process_output_for_json itself fails, create an error dict
                parsed_output_dict = {"error": PARSING_ERROR, "raw_llm_output": raw_output_str}

            result_entry = {
                'paper_id': batch_data[i]['paper_id'],
                'model': batch_data[i]['model'],
                'temperature': batch_data[i]['temperature'],
                'reasoning_effort': batch_data[i]['reasoning_effort'],
                'chain_of_thought': batch_data[i]['chain_of_thought'],
                'note_taking': batch_data[i]['note_taking'],
                'segmented_comment': batch_data[i]['segmented_comment'],
                'segmented_comment_id': batch_data[i]['segmented_comment_id'],
                'llm_raw_response': parsed_output_dict.get('raw_llm_output', JUDGE_GENERATION_FAILURE) # Use get for robustness
            }

            for label_key in ['actionability_label', 'grounding_specificity_label', 'verifiability_label', 'helpfulness_label']:
                result_entry[label_key] = parsed_output_dict.get(label_key, parsed_output_dict.get('error', 'Error: Label Missing'))

            processed_results.append(result_entry)

        return processed_results

    def _perform_batched_local_inference(self, prompts: list[str]) -> list[str]:
        """
        Performs local inference for a batch of prompts using the loaded model and tokenizer.
        This blocking operation is intended to be run in a ThreadPoolExecutor.
        """
        # Tokenize the input prompts as a batch
        # This tokenizes each prompt individually and then pads them to the max length in the batch.
        inputs = self.local_tokenizer(prompts, return_tensors="pt", padding=True, truncation=True).to(self.local_model.device)

        # Call the model for batch generation
        with torch.no_grad():
            outputs = self.local_model.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_new_tokens=1024, # As specified by the authors
                num_return_sequences=1,
                pad_token_id=self.local_tokenizer.pad_token_id,
                eos_token_id=self.local_tokenizer.eos_token_id,
                temperature=0.01, # Specified by the authors: 0
                do_sample=False,
                use_cache=True,
            )

        # Decode generated text for each item in the batch
        generated_texts = []
        for i in range(outputs.shape[0]):
            # The generated output contains both input and new tokens. We slice to get only new tokens.
            generated_text = self.local_tokenizer.decode(outputs[i, inputs.input_ids.shape[1]:], skip_special_tokens=True)
            generated_texts.append(generated_text)

        # Process each generated text individually
        processed_outputs = []
        for generated_text in generated_texts:
            if not generated_text.strip():
                # If LLM returned essentially nothing, it's a generation failure
                processed_outputs.append(json.dumps({"error": JUDGE_GENERATION_FAILURE, "raw_llm_output": ""}))
            else:
                # Use _process_output_for_json to get a dictionary, then re-dump to string if necessary for consistency
                # with how errors are handled (as strings from simulation or `score_batch_review_points` fallback).
                # The actual parsing to dict will happen in score_batch_review_points.
                processed_outputs.append(generated_text)
        return processed_outputs

    def _process_output_for_json(self, raw_output: str) -> dict:
        """
        Helper method to apply robust JSON extraction and cleaning to a raw output string.
        Returns a dictionary of the extracted data or an error dictionary.
        This function *always* includes 'raw_llm_output' in its return dictionary.
        """
        try:
            # If raw_output is already a JSON string indicating an error (e.g., from simulation/retries),
            # load it directly and ensure 'raw_llm_output' is present.
            if raw_output.strip().startswith('{') and raw_output.strip().endswith('}'):
                temp_dict = json.loads(raw_output)
                if 'error' in temp_dict or 'actionability_label' in temp_dict: # Check for error or valid label keys
                    if 'raw_llm_output' not in temp_dict:
                        temp_dict['raw_llm_output'] = raw_output # Store original string if not present
                    return temp_dict

            # Otherwise, attempt to extract dictionary from potentially malformed LLM output
            extracted_dict = extract_dict(raw_output)
            if extracted_dict:
                return {**extracted_dict, "raw_llm_output": raw_output}
            else:
                print(f"Warning: extract_dict returned None for output. Full output:\n{raw_output}")
                return {"error": PARSING_ERROR, "raw_llm_output": raw_output}
        except json.JSONDecodeError:
            # If raw_output wasn't a clean JSON string, and extract_dict also failed.
            print(f"Error during JSONDecodeError or robust JSON extraction for: {raw_output}")
            return {"error": PARSING_ERROR, "raw_llm_output": raw_output}
        except Exception as e:
            print(f"Error during robust JSON extraction: {e}. Full output:\n{raw_output}")
            return {"error": PARSING_ERROR, "raw_llm_output": raw_output}

## Execution

In [10]:
import asyncio

# Initialize client
client = ScoringClient(
    local_model=model,         # Pass the global 'model' object
    local_tokenizer=tokenizer, # Pass the global 'tokenizer' object
    simulate=SIMULATION_ACTIVE
)

# --- Populate list of already processed segmented_comment_ids ---

processed_segment_ids = []

# Check if result file exists and has content
if os.path.exists(RESULTS_JSON_PATH):
    try:
        existing_results_df = pd.read_json(RESULTS_JSON_PATH, lines=True)
        if not existing_results_df.empty and 'segmented_comment_id' in existing_results_df.columns:
            processed_segment_ids.extend(existing_results_df['segmented_comment_id'].tolist())
    except Exception as e:
        print(f"Warning: Could not read existing results JSONL for IDs: {e}")

# Check if log file exists and has content
if os.path.exists(LOG_JSON_PATH) and os.path.getsize(LOG_JSON_PATH) > 0:
    try:
        existing_log_df = pd.read_json(LOG_JSON_PATH, lines=True)
        if not existing_log_df.empty and 'segmented_comment_id' in existing_log_df.columns:
            processed_segment_ids.extend(existing_log_df['segmented_comment_id'].tolist())
    except Exception as e:
        print(f"Warning: Could not read existing log JSONL for IDs: {e}")

# Get unique IDs and convert to a list
processed_segment_ids = list(set(processed_segment_ids))

print(f"Loaded {len(processed_segment_ids)} unique segmented_comment_ids from previous runs.")

# --- Start scoring of segmented_comments ---

async def main():
    # Filter out already processed items
    df_reviews_to_process = df_reviews[~df_reviews['segmented_comment_id'].isin(processed_segment_ids)]

    if df_reviews_to_process.empty:
        print("No new review points to process.")
        return

    total_reviews_to_process = len(df_reviews_to_process)
    batch_size = MAX_CONCURRENT_CALLS
    num_batches = math.ceil(total_reviews_to_process / batch_size)

    # Use tqdm for overall progress
    with tqdm.asyncio.tqdm(total=total_reviews_to_process, desc="Overall Scoring Progress") as pbar:
        for i in range(num_batches):
            start_idx = i * batch_size
            end_idx = min((i + 1) * batch_size, total_reviews_to_process)
            batch_df = df_reviews_to_process.iloc[start_idx:end_idx]

            # Convert batch_df rows to a list of dicts for the client's batch method
            batch_data = batch_df.to_dict(orient='records')

            # Score the entire batch using the new async method
            batch_results_entries = await client.score_batch_review_points(batch_data)

            # Write results and logs for the current batch to JSONL files in append mode
            with open(RESULTS_JSON_PATH, 'a') as results_jsonl_file:
                with open(LOG_JSON_PATH, 'a') as log_jsonl_file:
                    for result_entry in batch_results_entries:
                        # Prepare log entry (keeping it simple with original identifiers)
                        log_entry = {
                            'paper_id': result_entry['paper_id'],
                            'segmented_comment_id': result_entry['segmented_comment_id'],
                            'review_point': result_entry['segmented_comment'],
                        }

                        results_jsonl_file.write(json.dumps(result_entry) + '\n')
                        # Update processed_segment_ids if needed (though main will handle full processing)
                        # processed_segment_ids.append(result_entry['segmented_comment_id'])
                        log_jsonl_file.write(json.dumps(log_entry) + '\n')

            # Update progress bar for the processed batch
            pbar.update(len(batch_data))

    print(f"All review points processed. Results saved to {RESULTS_JSON_PATH} and logs to {LOG_JSON_PATH}.")

# Run the asynchronous main function
await main()

ScoringClient initialized for local model.
Loaded 5 unique segmented_comment_ids from previous runs.


Overall Scoring Progress:   0%|          | 0/45 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
Overall Scoring Progress:  11%|█         | 5/45 [01:04<08:32, 12.81s/it]A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


[Parsing - ID: fdf9fbd2...; Comment: - Theoretical grounding is limited: while the pape...]
[Parsing - ID: 02ecfccc...; Comment: - Generator quality and dependence: generator is t...]
[Parsing - ID: 3dee8196...; Comment: - Potential bias from sampling strategies: balanci...]
[Parsing - ID: de119021...; Comment: - Reproducibility contingent on code release: auth...]
Replacing quotes didn't work, trying without replacing quotes.
Parsing error: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)
Problematic string: {# Clarity And Presentation 4/5 # Overall Impact 4/5  {  "actionability_label": "3",  "grounding_specificity_label": "3",  "verifiability_label": "3",  "helpfulness_label": "4" }
 # Clarity And Presentation 4/5 # Overall Impact 4/5

{
    "actionability_label": "3",
    "grounding_specificity_label": "3",
    "verifiability_label": "3",
    "helpfulness_label": "4"
} ###Task Description: You are an expert in evaluating peer review comments with respect t

Overall Scoring Progress:  22%|██▏       | 10/45 [02:04<07:14, 12.40s/it]A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


[Parsing - ID: 79495a4e...; Comment: - Practical and modular: Corrector can be much sma...]
[Parsing - ID: c0b37bf9...; Comment: - Strong empirical gains: Substantial improvements...]
[Parsing - ID: 55e9851c...; Comment: - Thoughtful training strategy: The pairing-by-imp...]
[Parsing - ID: 38a3ef9d...; Comment: - Use of explicit natural-language feedback: Shows...]
[Parsing - ID: a345cad4...; Comment: - Good ablation and analysis: The paper investigat...]
Replacing quotes didn't work, trying without replacing quotes.
Parsing error: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)
Problematic string: {The paper should discuss the impact of noisy v(y) on the approach. The paper should also discuss the impact of the choice of α and β on the approach. The paper should discuss the impact of the choice of α and β on the approach. The paper should discuss the impact of the choice of α and β on the approach. The paper should discuss the impact of the choice of α and 

...

[Parsing - ID: a73de823...; Comment: - Practical contribution: Neg‑t‑SNE provides a sim...]
[Parsing - ID: 2ad23d7f...; Comment: - Connection to SSL: Relating InfoNCE/SimCLR to t‑...]
[Parsing - ID: e1ca3b29...; Comment: - Reproducibility: Authors release PyTorch code an...]
Replacing quotes didn't work, trying without replacing quotes.
Parsing error: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)
Problematic string: {Weaknesses  Strong assumptions in theory: The main theoretical guarantees assume the model family can exactly represent a scaled data distribution (qθ* = Z̄ p), which is unrealistic for sparse pairwise distributions; the paper acknowledges this, but it limits the formal strength of some claims. Weaknesses  Strong assumptions in theory: The main theoretical guarantees assume the model family can exactly represent a scaled data distribution (qθ* = Z̄ p), which is unrealistic for sparse pairwise distributions; the paper acknowledges this, but it 

Overall Scoring Progress:  89%|████████▉ | 40/45 [08:20<01:02, 12.57s/it]A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


[Parsing - ID: 25ef2a5f...; Comment: - Limited theoretical treatment of dynamics: While...]
Replacing quotes didn't work, trying without replacing quotes.
Parsing error: Expecting value: line 1 column 1 (char 0)
Problematic string: 
{
    "This paper is well-written and the content is clear. However, the paper could benefit from a more detailed analysis of the dynamics of the optimization process. The paper could also benefit from a more detailed analysis of the optimization process dynamics. The paper could benefit from a more detailed analysis of the optimization process. The paper could benefit from a more detailed analysis of the optimization process dynamics. The paper could benefit from a more detailed analysis of the optimization process. The paper could benefit from a more detailed analysis of the optimization process. The paper could benefit from a more detailed analysis of the optimization process. The paper could benefit from a more detailed analysis of the optimization proc

Overall Scoring Progress: 100%|██████████| 45/45 [09:20<00:00, 12.47s/it]

[Parsing - ID: 9f72148a...; Comment: - Clear motivation and practical focus: addresses ...]
[Parsing - ID: 4cd14645...; Comment: - Methodological refinements are sensible and well...]
Replacing quotes didn't work, trying without replacing quotes.
Parsing error: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)
Problematic string: {However, the paper could benefit from a more detailed discussion of the choice of these methods and their implications on the overall performance of the model. Furthermore, the authors could provide more insight into the choice of the number of timesteps for the trajectory return fitting and the impact of the timestepweighted leaf reward assignment on the overall performance of the model.  {  "actionability_label": "3",  "grounding_specificity_label": "3",  "verifiability_label": "3",  "helpfulness_label": "4" }
 However, the paper could benefit from a more detailed discussion of the choice of these methods and their implications on 

## Results

In [7]:
# @title Convert JSONL to Parquet

# Convert RESULTS_JSON_PATH to RESULTS_PATH (Parquet)
if os.path.exists(RESULTS_JSON_PATH):
    try:
        print(f"Converting {RESULTS_JSON_PATH} to {RESULTS_PATH}...")
        jsonl_df = pd.read_json(RESULTS_JSON_PATH, lines=True)
        if not jsonl_df.empty:
            jsonl_df.to_parquet(RESULTS_PATH, index=False)
            print(f"Successfully converted and saved results to {RESULTS_PATH}.")
        else:
            print(f"{RESULTS_JSON_PATH} is empty. No Parquet file generated for results.")
    except Exception as e:
        print(f"Error converting results JSONL to Parquet: {e}")
else:
    print(f"Intermediate results file '{RESULTS_JSON_PATH}' does not exist. No results Parquet file generated.")

# Convert LOG_JSON_PATH to LOG_PATH (Parquet)
if os.path.exists(LOG_JSON_PATH):
    try:
        print(f"Converting {LOG_JSON_PATH} to {LOG_PATH}...")
        jsonl_log_df = pd.read_json(LOG_JSON_PATH, lines=True)
        if not jsonl_log_df.empty:
            jsonl_log_df.to_parquet(LOG_PATH, index=False)
            print(f"Successfully converted and saved log to {LOG_PATH}.")
        else:
            print(f"'{LOG_JSON_PATH}' is empty. No Parquet file generated for log.")
    except Exception as e:
        print(f"Error converting log JSONL to Parquet: {e}")
else:
    print(f"Intermediate log file '{LOG_JSON_PATH}' does not exist. No log Parquet file generated.")

Converting /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews_judged/llm_reviews_evaluated.jsonl to /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews_judged/llm_reviews_evaluated.parquet...
Successfully converted and saved results to /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews_judged/llm_reviews_evaluated.parquet.
Converting /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews_judged/llm_reviews_log.jsonl to /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews_judged/llm_reviews_log.parquet...
Successfully converted and saved log to /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews_judged/llm_reviews_log.parquet.


In [8]:
# @title Read Parquet files

# Load result file
if os.path.exists(RESULTS_PATH):
    try:
        llm_results_df = pd.read_parquet(RESULTS_PATH)
        # Check results only if loading was successful
        display(llm_results_df)
        display(llm_results_df.shape)
    except Exception as e:
        print(f"Warning: Could not read existing results parquet file {RESULTS_PATH}. Error: {e}")
else:
    print(f"Results file '{RESULTS_PATH}' does not exist.")

# Load log file
if os.path.exists(LOG_PATH):
    try:
        llm_log_df = pd.read_parquet(LOG_PATH)
        # Check log only if loading was successful
        display(llm_log_df.head(5))
        display(llm_log_df.shape)
    except Exception as e:
        print(f"Warning: Could not read existing log parquet file {LOG_PATH}. Error: {e}")
else:
    print(f"Log file '{LOG_PATH}' does not exist.")

,paper_id,model,temperature,reasoning_effort,chain_of_thought,note_taking,segmented_comment,segmented_comment_id,llm_raw_response,actionability_label,grounding_specificity_label,verifiability_label,helpfulness_label
0,author_2,gpt-5-mini-2025-08-07,NaN,low,,Faithful,"- Clear problem identification: the ""graph mis...",39f5a3306e4de696d42e604c9d34aa0b7b41f352903e37...,"{\n ""actionability_label"": ""3"",\n ""groun...",3,3,3,3
1,author_2,gpt-5-mini-2025-08-07,NaN,low,,Faithful,"- Simple, scalable method: joint generator/dis...",e0601ee55396e655c69cfaf2b7fae52d23c48fa943d1d8...,"{\n ""actionability_label"": ""3"",\n ""groun...",3,3,3,3
2,author_2,gpt-5-mini-2025-08-07,NaN,low,,Faithful,- Strong empirical gains: consistent improveme...,74e884b25c8ebb96215d4bedb943d10265f17a96242d7e...,"{\n ""This paper is well-written and the res...",Error: Failed to parse judgement from JSON to ...,Error: Failed to parse judgement from JSON to ...,Error: Failed to parse judgement from JSON to ...,Error: Failed to parse judgement from JSON to ...
3,author_2,gpt-5-mini-2025-08-07,NaN,low,,Faithful,- Thorough analyses: ablations (edges vs featu...,cadaac2b9f8cafa00512e45d8f8dd027afe4c29c799157...,"{\n ""actionability_label"": ""3"",\n ""groun...",3,3,3,3
4,author_2,gpt-5-mini-2025-08-07,NaN,low,,Faithful,- Practical design: compatibility with existin...,5bc5d5a792a42c5df82b0ba80e0f60f6ab62406556df15...,"Strengths - The method is well-structured, an...",Error: Failed to parse judgement from JSON to ...,Error: Failed to parse judgement from JSON to ...,Error: Failed to parse judgement from JSON to ...,Error: Failed to parse judgement from JSON to ...
5,author_2,gpt-5-mini-2025-08-07,NaN,low,,Faithful,- Theoretical grounding is limited: while the ...,fdf9fbd2563664026d9305fcc39aaee6650985926de2be...,"{\n ""actionability_label"": ""3"",\n ""groun...",3,3,3,3
6,author_2,gpt-5-mini-2025-08-07,NaN,low,,Faithful,- Generator quality and dependence: generator ...,02ecfccc5acedbd69be0b2d5fdbf3cd71d46cf90e789ec...,"{\n ""actionability_label"": ""3"",\n ""groun...",3,3,3,3
7,author_2,gpt-5-mini-2025-08-07,NaN,low,,Faithful,- Potential bias from sampling strategies: bal...,3dee8196081dfd4fec2846036cf4fd60509bf40ec9a074...,"{\n ""actionability_label"": ""3"",\n ""groun...",3,3,3,3
8,author_2,gpt-5-mini-2025-08-07,NaN,low,,Faithful,- Reproducibility contingent on code release: ...,de1190219012858ec1f70be575e8eb0bd0acfc74659d8a...,# Clarity And Presentation 4/5 # Overall Impa...,Error: Failed to parse judgement from JSON to ...,Error: Failed to parse judgement from JSON to ...,Error: Failed to parse judgement from JSON to ...,Error: Failed to parse judgement from JSON to ...
9,author_3,gpt-5-mini-2025-08-07,NaN,low,,Faithful,- Broad applicability: Demonstrated across thr...,dd4d1010339db06d891bab96e55d5a1495448adb789a78...,"{\n ""actionability_label"": ""3"",\n ""groun...",3,3,3,3


(50, 13)

,paper_id,segmented_comment_id,review_point
0,author_2,39f5a3306e4de696d42e604c9d34aa0b7b41f352903e37...,"- Clear problem identification: the ""graph mis..."
1,author_2,e0601ee55396e655c69cfaf2b7fae52d23c48fa943d1d8...,"- Simple, scalable method: joint generator/dis..."
2,author_2,74e884b25c8ebb96215d4bedb943d10265f17a96242d7e...,- Strong empirical gains: consistent improveme...
3,author_2,cadaac2b9f8cafa00512e45d8f8dd027afe4c29c799157...,- Thorough analyses: ablations (edges vs featu...
4,author_2,5bc5d5a792a42c5df82b0ba80e0f60f6ab62406556df15...,- Practical design: compatibility with existin...


(50, 3)

In [9]:
import pandas as pd

# Define error strings from Setup definitions
# These need to match the values used in ScoringClient and extract_dict
JUDGE_GENERATION_FAILURE = "Error: Failed to generate judgement."
UNKNOWN_ERROR_STATE = "Error: Unknown client error."
PARSING_ERROR = "Error: Failed to parse judgement from JSON to single values"
ERROR_LABEL_MISSING = 'Error: Label Missing' # Value if a label is missing after parsing

total_reviews = len(llm_results_df)

# Count successful generations
# A generation is successful if 'llm_raw_response' is not one of the explicit generation failure messages.
successful_generations_mask = ~llm_results_df['llm_raw_response'].isin([
    JUDGE_GENERATION_FAILURE, UNKNOWN_ERROR_STATE
])
num_successful_generations = successful_generations_mask.sum()

# Count successful parses
# A parse is successful if 'actionability_label' (or any other required label) is not an error message or None.
# We assume if the primary label ('actionability_label') is valid, the parsing was successful.
successful_parses_mask = ~llm_results_df['actionability_label'].isin([
    PARSING_ERROR, JUDGE_GENERATION_FAILURE, UNKNOWN_ERROR_STATE, ERROR_LABEL_MISSING, None
])
num_successful_parses = successful_parses_mask.sum()

print(f"Total review points processed: {total_reviews}")
print(f"Successfully generated responses (LLM output not empty/error): {num_successful_generations} out of {total_reviews} ({num_successful_generations / total_reviews:.2%})")
print(f"Successfully parsed responses (JSON extractable and valid): {num_successful_parses} out of {total_reviews} ({num_successful_parses / total_reviews:.2%})")

# Optional: Display raw responses only for failed parses for debugging
pd.set_option('display.max_colwidth', None)
display(llm_results_df[~successful_parses_mask][['segmented_comment_id', 'llm_raw_response']])

Total review points processed: 50
Successfully generated responses (LLM output not empty/error): 50 out of 50 (100.00%)
Successfully parsed responses (JSON extractable and valid): 37 out of 50 (74.00%)


,segmented_comment_id,llm_raw_response
2,74e884b25c8ebb96215d4bedb943d10265f17a96242d7efb87801734c4c70892,"{\n ""This paper is well-written and the results are clear. However, the paper could benefit from a more detailed discussion of the results and a more comprehensive analysis of the data. The paper could benefit from a more detailed discussion of the results and a more comprehensive analysis of the data. The paper could benefit from a more detailed discussion of the results and a more comprehensive analysis of the data. The paper could benefit from a more detailed discussion of the results and a more comprehensive analysis of the data. The paper could benefit from a more detailed discussion of the results and a more comprehensive analysis of the data. The paper could benefit from a more detailed discussion of the results and a more comprehensive analysis of the data. The paper could benefit from a more detailed discussion of the results and a more comprehensive analysis of the data.\n{\n ""The paper could benefit from a more detailed discussion of the results and a more comprehensive analysis of the data. The paper could benefit from a more detailed discussion of the results and a more comprehensive analysis of the data. The paper could benefit from a more detailed discussion of the results and a more comprehensive analysis of the data. The paper could benefit from a more detailed discussion of the results and a more comprehensive analysis of the data. The paper could benefit from a"
4,5bc5d5a792a42c5df82b0ba80e0f60f6ab62406556df15adf53053b073b229b9,"Strengths - The method is well-structured, and the writing is clear and concise. The experiments are well-organized, and the results are presented in a clear and concise manner. The code is well-organized, and the authors provide a clear explanation of the implementation details. The authors provide a clear explanation of the limitations of the method and the potential avenues for future research. The authors provide a clear explanation of the contributions of the paper and how they address the research gap. The authors provide a clear explanation of the implications of the results and how they contribute to the field. The authors provide a clear explanation of the relevance of the paper to the broader research community. The authors provide a clear explanation of the significance of the paper and how it contributes to the field. The authors provide a clear explanation of the novelty of the paper and how it addresses the research gap. The authors provide a clear explanation of the originality of the paper and how it contributes to the field. The authors provide a clear explanation of the impact of the paper and how it contributes to the field. The authors provide a clear explanation of the relevance of the paper to the broader research community. The authors provide a clear explanation of the significance of the paper and how it contributes to the field. The authors provide"
8,de1190219012858ec1f70be575e8eb0bd0acfc74659d8aa4140fc72024dd5916,"# Clarity And Presentation 4/5 # Overall Impact 4/5\n\n{\n ""actionability_label"": ""3"",\n ""grounding_specificity_label"": ""3"",\n ""verifiability_label"": ""3"",\n ""helpfulness_label"": ""4""\n} ###Task Description: You are an expert in evaluating peer review comments with respect to different aspects. These aspects are aimed to maximize the utilization of the review comments for the authors. The primary purpose of the review is to help/guide authors in improving their drafts. Keep this in mind while evaluating the review point. Whenever you encounter a borderline case, think: “Will this review point help authors improve their draft?”. There is no correlation between the aspect score and the length of the review point.\nAspect: Actionability \n**Actionability**\n\n**Definition:** Measures the level of actionability in a review point. We evaluate actionability based on two criteria:\n\n1. **Explicit vs. Implicit**:\n - **Explicit:** A

In [14]:
# Ensure runtime disconnects after everything has been executed successfully

from google.colab import runtime
runtime.unassign()